In [1]:
import ssl
import certifi
ssl._create_default_https_context = ssl._create_unverified_context
from llama_index.core import SimpleDirectoryReader
import os
from llama_index.core.node_parser import SentenceSplitter
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_list = [
    "../library-app/docs",
    "../library-app/.kiro", 
    "../library-app/.windsurf"
]
documents = []

for path in path_list:
    if os.path.exists(path):
        try:
            reader = SimpleDirectoryReader(
                input_dir=path, 
                recursive=True, 
                required_exts=[".md"],
                exclude_hidden=False 
            )
            data = reader.load_data()
            documents.extend(data)
        except ValueError:
            print(f"❓ התיקייה {path} נמצאה, אך לא נמצאו בה קבצי .md")
    else:
        print(f"❌ שגיאה: הנתיב {path} לא נמצא. ודאי ששם התיקייה מדויק!")



In [3]:
node_parser = SentenceSplitter(chunk_size=1024, chunk_overlap=20)
nodes = node_parser.get_nodes_from_documents(documents)

In [4]:
from dotenv import load_dotenv
import os

# הפקודה הזו הולכת לקובץ ה-.env שלך וטוענת את המשתנים לזיכרון
load_dotenv() 

# בדיקה קטנה כדי לראות שזה עובד (ידפיס True אם המפתח נטען)
print(f"Is API Key loaded? {bool(os.getenv('COHERE_API_KEY'))}")

Is API Key loaded? True


In [5]:
from llama_index.embeddings.cohere import CohereEmbedding
import os
from llama_index.core import Settings
embed_model = CohereEmbedding(
    api_key=os.getenv("COHERE_API_KEY"),
    model_name="embed-multilingual-v3.0",
)
Settings.embed_model = embed_model

In [6]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pinecone_index = pc.Index("agentic-coding-index")
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes, storage_context=storage_context)

2026-02-26 12:23:35,374 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
Upserted vectors: 100%|██████████| 6/6 [00:02<00:00,  2.62it/s]


# Part 1: Basic RAG with Gradio Interface

In [7]:
from llama_index.llms.cohere import Cohere
Settings.llm = Cohere(
    api_key=os.getenv("COHERE_API_KEY"), 
    model="command-a-03-2025" 
)

In [8]:
import gradio as gr

query_engine = index.as_query_engine()

def predict(message, history):
    response = query_engine.query(message)
    return str(response)

demo = gr.ChatInterface(
    fn=predict, 
    title="Agentic RAG Assistant (Local Mode)",
    description="שאל אותי כל דבר על התיעוד של הפרויקט!",
)

demo.launch(share=False, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860


2026-02-26 12:23:48,048 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-02-26 12:23:48,115 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-02-26 12:23:48,390 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


# Part 2: Event-Driven RAG Workflow

In [9]:
from llama_index.core.workflow import Event
from llama_index.core.schema import NodeWithScore

class RetrievedEvent(Event):
    nodes: list[NodeWithScore]
    query: str

class ValidationEvent(Event):
    query: str
    nodes: list[NodeWithScore]

class RetryRetrievalEvent(Event):
    query: str



In [10]:
from llama_index.core import Settings, get_response_synthesizer
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step, Context


Settings.llm = Cohere(api_key=os.getenv("COHERE_API_KEY"), model="command-a-03-2025")
response_synthesizer = get_response_synthesizer(response_mode="compact")

2026-02-26 12:23:48,648 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


In [11]:
class RAGWorkflow(Workflow):
    retries = 0 
    @step
    async def ingest_and_validate(self, ctx: Context, ev: StartEvent) -> RetrievedEvent | StopEvent:
        query = ev.get("query")
        if not query or query.strip() == "":
            return StopEvent(result="⚠️ השאילתה ריקה, נא להזין שאלה.")
        
        print(f"🔎 [ניסיון {self.retries + 1}] מבצע Retrieval עבור: {query}")
        nodes = retriever.retrieve(query)
        return RetrievedEvent(nodes=nodes, query=query)

    @step
    async def validate_results(self, ctx: Context, ev: RetrievedEvent) -> ValidationEvent | RetryRetrievalEvent | StopEvent:
        top_score = ev.nodes[0].score if ev.nodes and ev.nodes[0].score else 0
        
        if top_score < 0.5 and self.retries < 2:
            print(f"🔄 Confidence גבולי ({top_score:.2f}). מנתב לשיפור שאילתה...")
            self.retries += 1
            return RetryRetrievalEvent(query=f"Detailed technical information about {ev.query}")
        
        if not ev.nodes or top_score < 0.35:
            self.retries = 0 # איפוס לריצה הבאה
            return StopEvent(result="❌ לא נמצא מידע מהימן מספיק במסמכים.")

        print(f"✅ ולידציה עברה (Score: {top_score:.2f})")
        self.retries = 0 
        return ValidationEvent(nodes=ev.nodes, query=ev.query)

    @step
    async def handle_retry(self, ctx: Context, ev: RetryRetrievalEvent) -> StartEvent:
        return StartEvent(query=ev.query)
    
    @step
    async def synthesize(self, ctx: Context, ev: ValidationEvent) -> StopEvent:
        # בדיקה: האם המשתמש ביקש JSON או תשובה רגילה?
        # (הסוכן של שלב ג' ידע להוסיף את המילה JSON לבקשה)
        if "json" in ev.query.lower() or "extract" in ev.query.lower():
            print("✍️ מחלץ מידע למבנה JSON מובנה (שלב ג')...")
            structured_llm = Settings.llm
            response = structured_llm.structured_predict(
                ProjectDataSchema, 
                prompt_template="Extract technical data from: {context_str}",
                context_str="\n".join([n.text for n in ev.nodes])
            )
            return StopEvent(result=response.model_dump_json(indent=2))
        
        else:
            print("✍️ מנסח תשובה טקסטואלית רגילה (שלב ב')...")
            # שימוש בסינתיסייזר הרגיל משלב א' וב'
            response = response_synthesizer.synthesize(query=ev.query, nodes=ev.nodes)
            return StopEvent(result=str(response))


In [12]:

# אתחול ה-retriever - נוצר מתוך ה-index (נמצא בתא קודם)
retriever = index.as_retriever(similarity_top_k=3)

rag_workflow = RAGWorkflow(timeout=60, verbose=True)
result = await rag_workflow.run(query="What is the primary color of the design system ??")
print("-" * 30)
print(f"🤖 תשובה מה-Workflow: {result}")

2026-02-26 12:23:49,348 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


🔎 [ניסיון 1] מבצע Retrieval עבור: What is the primary color of the design system ??


2026-02-26 12:23:49,645 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


🔄 Confidence גבולי (0.47). מנתב לשיפור שאילתה...
🔎 [ניסיון 2] מבצע Retrieval עבור: Detailed technical information about What is the primary color of the design system ??


2026-02-26 12:23:50,797 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


✅ ולידציה עברה (Score: 0.50)
✍️ מנסח תשובה טקסטואלית רגילה (שלב ב')...


2026-02-26 12:23:52,082 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


------------------------------
🤖 תשובה מה-Workflow: The primary color of the design system is Ocean Blue, represented by the hexadecimal code #0077be.


In [13]:
async def predict(message, history):
    workflow = RAGWorkflow(timeout=120)
    result = await workflow.run(query=message)
    return str(result)

demo = gr.ChatInterface(
    fn=predict,
    title="Kiro Agentic RAG - Event Driven",
    description="מערכת RAG מבוססת אירועים עם שיפור שאילתות אוטומטי (שלב ב')"
)

demo.launch(share=True, debug=True)

2026-02-26 12:23:52,342 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861


2026-02-26 12:23:53,099 - INFO - HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-02-26 12:23:53,398 - INFO - HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"
2026-02-26 12:23:53,714 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-02-26 12:23:55,482 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026-02-26 12:23:56,484 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-error-analytics "HTTP/1.1 200 OK"
2026-02-26 12:23:56,567 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


Keyboard interruption in main thread... closing server.


# Part 3: Agentic RAG (Final Stage)

In [15]:
from pydantic import BaseModel, Field
from typing import List

class DecisionItem(BaseModel):
    id: str = Field(description="Unique ID like dec-001")
    title: str = Field(description="Brief title of the decision")
    summary: str = Field(description="What was decided?")
    file: str = Field(description="The source file name")

class ProjectDataSchema(BaseModel):
    decisions: List[DecisionItem]
    rules: List[dict]
    warnings: List[dict]

In [16]:
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import FunctionTool

async def technical_research_tool(query: str) -> str:
    """
    Search for technical specifications, UI design decisions, and JSON extraction.
    Use this tool ONLY for technical questions about the library app or when JSON is requested.
    """
    workflow = RAGWorkflow(timeout=120)
    result = await workflow.run(query=query)
    return str(result)

research_tool = FunctionTool.from_defaults(
    async_fn=technical_research_tool,
    name="technical_research",
    description="Extracts technical details or JSON from project documents."
)

from llama_index.core.agent import ReActAgent

agent = ReActAgent(
    tools=[research_tool], 
    llm=Settings.llm, 
    verbose=True
)


In [17]:
from llama_index.core.llms import ChatMessage

async def agent_logic(message: str):
    decision_prompt = f"""
    You are a router. Classify the user message:
    
    1. If it's a greeting, personal question ("who are you"), or general knowledge (France, math, etc.) -> Output 'GENERAL'.
    2. If it's about the library app, UI, Kiro, JSON, or tech specs -> Output 'TECHNICAL'.
    
    Message: "{message}"
    Answer ONLY one word: GENERAL or TECHNICAL.
    """
    
    msg = [ChatMessage(role="user", content=decision_prompt)]
    decision_res = str(Settings.llm.chat(msg)).strip().upper()
    
    is_technical = "TECHNICAL" in decision_res and "GENERAL" not in decision_res

    if is_technical:
        print(f"🔍 ניתוב למסלול TECHNICAL (RAG)")
        wf = RAGWorkflow(timeout=120)
        return await wf.run(query=message)
    else:
        print(f"💬 ניתוב למסלול GENERAL (ידע כללי)")
        msg_gen = [ChatMessage(role="user", content=message)]
        response = Settings.llm.chat(msg_gen)
        return str(response)

In [ ]:
import gradio as gr

async def predict(message, history):
    print(f"\n--- בקשה חדשה מהממשק: {message} ---")
    result = await agent_logic(message)
    return str(result)

demo = gr.ChatInterface(
    fn=predict, 
    title="Kiro Agentic System - Final Stage",
    description="סוכן חכם: יודע גם לשלוף מידע טכני ומובנה (JSON) ממסמכי הפרויקט, וגם לנהל שיחת חולין."
)

demo.launch(share=False, debug=True)

2026-02-26 12:24:22,301 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861


2026-02-26 12:24:22,648 - INFO - HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-02-26 12:24:22,674 - INFO - HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


2026-02-26 12:24:22,888 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-02-26 12:24:23,464 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


...

2026-02-26 12:24:49,338 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


🔄 Confidence גבולי (0.44). מנתב לשיפור שאילתה...
🔎 [ניסיון 3] מבצע Retrieval עבור: Detailed technical information about Detailed technical information about what the [roject do?


2026-02-26 12:24:50,114 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


✅ ולידציה עברה (Score: 0.46)
✍️ מנסח תשובה טקסטואלית רגילה (שלב ב')...


2026-02-26 12:24:57,471 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
